# Test synchronisation Docs → Grist
Notebook de test pour valider chaque étape avant la synchro complète.

In [1]:
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Connexion au client Docs
Renseigne `DOCS_SESSION_ID` et `DOCS_CSRF_TOKEN` dans le fichier `.env` avant de lancer cette cellule.

In [2]:
import os
from docs_client import DocsClient

docs = DocsClient(
    base_url='https://docs.numerique.gouv.fr',
    session_id=os.environ.get('DOCS_SESSION_ID'),
    csrf_token=os.environ.get('DOCS_CSRF_TOKEN'),
)
print('DocsClient initialisé.')

DocsClient initialisé.


## 2. Récupération du tree
Remplace l'URL ci-dessous par celle de ton document racine.

In [3]:
ROOT_URL = 'https://docs.numerique.gouv.fr/docs/21bb0cdd-638f-4a50-8f3a-03616716771e/'

doc_id = DocsClient.extract_doc_id(ROOT_URL)
tree = docs.get_tree(doc_id)

print(f"Racine : {tree['title']}")
print(f"Enfants directs : {len(tree.get('children', []))}")

Racine : Cartographie Stratégique du Bruit - Guide Méthodologique
Enfants directs : 13


## 2b. Diagnostic émojis — comparaison des formats

Teste un document qui contient des émojis pour voir ce que retourne chacun des 3 formats de l'API.
On compare markdown / HTML / JSON pour choisir le plus fiable.

In [4]:
# ── Choisir un document qui contient des émojis ──────────────────────────────
# Remplace l'URL ci-dessous par un doc de ton espace qui a des émojis dans le corps
EMOJI_DOC_URL = 'https://docs.numerique.gouv.fr/docs/242c7d7f-b715-48bc-aa06-677a8dfc9ad0/'

emoji_doc_id = DocsClient.extract_doc_id(EMOJI_DOC_URL)

# Vérifie l'émoji dans le titre
meta = docs.get_document(emoji_doc_id)
titre_complet = meta.get('title', '')
emoji_titre, titre_propre = DocsClient.extract_emoji_from_title(titre_complet)

print(f"Titre complet  : {titre_complet!r}")
print(f"Émoji extrait  : {emoji_titre!r}")
print(f"Titre propre   : {titre_propre!r}")
print()

# Récupère les 3 formats et compare
print("── Récupération des 3 formats ─────────────────────────────────────────")
formats = docs.get_content_all_formats(emoji_doc_id)

for fmt in ("markdown", "html", "json"):
    err = formats.get(f"{fmt}_error")
    content = formats.get(fmt)
    if err:
        print(f"\n[{fmt.upper()}] ❌ Erreur : {err}")
    elif content is None:
        print(f"\n[{fmt.upper()}] ⚠️  Contenu vide")
    else:
        n_emojis = DocsClient.count_emojis(str(content))
        preview = str(content)[:300].replace("\n", "↵")
        print(f"\n[{fmt.upper()}] ✓ {len(str(content))} caractères, {n_emojis} émoji(s)")
        print(f"  Aperçu : {preview}…")

Titre complet  : '🌟 Guide'
Émoji extrait  : '🌟'
Titre propre   : 'Guide'

── Récupération des 3 formats ─────────────────────────────────────────

[MARKDOWN] ✓ 304 caractères, 0 émoji(s)
  Aperçu : Ce guide méthodologique est construit autour de la structure suivante :↵↵Quelques éléments de contexte et de vocabulaire :↵↵*↵↵*↵↵*↵↵*↵↵*↵↵*↵↵Les utilisés :↵↵* Modélisation du bruit :↵↵* Stockage et gestion des données :↵↵* Exécution des calculs :↵↵Les données d'entrée :↵↵*↵↵*↵↵*↵↵*↵↵*↵↵*↵↵*↵↵*↵↵*↵↵…

[HTML] ✓ 1108 caractères, 0 émoji(s)
  Aperçu : <p>Ce guide méthodologique est construit autour de la structure suivante :</p><p></p><p>Quelques éléments de contexte et de vocabulaire :</p><ul><li><p class="bn-inline-content"></p></li><li><p class="bn-inline-content"></p></li><li><p class="bn-inline-content"></p></li><li><p class="bn-inline-conte…

[JSON] ✓ 6345 caractères, 0 émoji(s)
  Aperçu : [{'id': 'initialBlockId', 'type': 'paragraph', 'props': {'backgroundColor': 'default', 'textColor': 

## 2c. Test sur le document vide connu

Vérifie que le document vide (`2ca8fee3-…`) est désormais traité sans lever d'exception.

In [5]:
EMPTY_DOC_URL = 'https://docs.numerique.gouv.fr/docs/2ca8fee3-93ab-45b6-ba1b-f9abbae398ef/'
empty_doc_id = DocsClient.extract_doc_id(EMPTY_DOC_URL)

meta_empty = docs.get_document(empty_doc_id)
print(f"Titre          : {meta_empty.get('title')!r}")
print(f"numchild       : {meta_empty.get('numchild', '?')}")
print()

# Récupère les 3 formats — doit renvoyer "" sans exception
formats_empty = docs.get_content_all_formats(empty_doc_id)

for fmt in ("markdown", "html", "json"):
    err   = formats_empty.get(f"{fmt}_error")
    value = formats_empty.get(fmt)
    if err:
        print(f"[{fmt.upper()}] ❌ Erreur  : {err}")
    elif value == "" or value is None:
        print(f"[{fmt.upper()}] ✓ Vide (attendu pour un document sans contenu)")
    else:
        print(f"[{fmt.upper()}] ? Contenu inattendu : {str(value)[:100]!r}")

Titre          : '🛤️ Voie ferroviaire'
numchild       : 0

    ⚠️  Y-Provider 500 pour 2ca8fee3-93ab-45b6-ba1b-f9abbae398ef [markdown] — contenu non convertible, traité comme vide.
    ⚠️  Y-Provider 500 pour 2ca8fee3-93ab-45b6-ba1b-f9abbae398ef [html] — contenu non convertible, traité comme vide.
    ⚠️  Y-Provider 500 pour 2ca8fee3-93ab-45b6-ba1b-f9abbae398ef [json] — contenu non convertible, traité comme vide.
[MARKDOWN] ✓ Vide (attendu pour un document sans contenu)
[HTML] ✓ Vide (attendu pour un document sans contenu)
[JSON] ✓ Vide (attendu pour un document sans contenu)


## 3. Aplatissement du tree en records Grist

In [ ]:

from grist_client import GristClient
from docs_client import DocsClient
import requests

grist = GristClient()

# ── Colonnes existantes dans la table Grist ───────────────────────────────────
GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}
# GRIST_COLUMNS = None  # ← décommente pour envoyer tous les champs sans filtrage

# ── Taille maximale du champ contenu (caractères) ────────────────────────────
MAX_CONTENU = 100_000

# ── Langages de code à neutraliser avant envoi (risque WAF/Incapsula) ────────
# On cible uniquement les langages qui déclenchent des faux positifs SQL/injection.
# NE PAS inclure 'callout', 'python', etc. — ce sont des blocs de contenu légitimes
# sérialisés par BlockNote et leur suppression ferait perdre du contenu.
WAF_SQL_LANGUAGES = ['sql', 'postgresql', 'postgres', 'psql', 'plpgsql',
                     'mysql', 'sqlite', 'tsql', 'mariadb']

def filter_record(record):
    fields = {k: v for k, v in record['fields'].items()
              if GRIST_COLUMNS is None or k in GRIST_COLUMNS}
    if 'contenu' in fields and isinstance(fields['contenu'], str):
        # Neutralise uniquement les blocs SQL (déclencheurs WAF)
        fields['contenu'] = DocsClient.strip_code_blocks(
            fields['contenu'], languages=WAF_SQL_LANGUAGES
        )
        if len(fields['contenu']) > MAX_CONTENU:
            fields['contenu'] = fields['contenu'][:MAX_CONTENU] + '\n[…contenu tronqué]'
    return {'fields': fields}

ok, ko = [], []

for i, record in enumerate(records, start=1):
    titre = record['fields'].get('titre') or record['fields'].get('titre_propre', '?')
    filtered = filter_record(record)
    contenu_len = len(filtered['fields'].get('contenu', ''))
    try:
        grist.add_records('Chapitres', [filtered])
        ok.append(i)
        print(f"  [{i:>3}/{len(records)}] ✓  {titre}  ({contenu_len} car.)")
    except requests.HTTPError as e:
        body = e.response.text if e.response is not None else '(pas de réponse)'
        ko.append({'index': i, 'titre': titre, 'erreur': str(e), 'body': body, 'record': record})
        print(f"  [{i:>3}/{len(records)}] ✗  {titre}  ({contenu_len} car.)")
        print(f"          → {e}")
        print(f"          Corps Grist : {body[:300]}")
    except Exception as e:
        ko.append({'index': i, 'titre': titre, 'erreur': str(e), 'body': '', 'record': record})
        print(f"  [{i:>3}/{len(records)}] ✗  {titre}")
        print(f"          → {e}")

print(f"\n── Résumé ──────────────────────────────────────────")
print(f"  Succès : {len(ok)}")
print(f"  Échecs : {len(ko)}")
if ko:
    print("\n  Détail des échecs :")
    for item in ko:
        print(f"    [{item['index']}] {item['titre']!r} → {item['erreur']}")
        if item.get('body'):
            print(f"             Corps : {item['body'][:200]}")


[racine] Cartographie Stratégique du Bruit - Guide Méthodologique
  [1] 🌟 Guide
    [1.1] 📜 ​Rapportage
    ✓ 1 émoji(s) dans le contenu [markdown]
    [1.2] 🛣️ GITT
    ✓ 1 émoji(s) dans le contenu [markdown]
    [1.3] 🗺️ CBS
    [1.4] 🔠 Glossaire
    [1.5] 📚 Bibliographie associée
    [1.6] ℹ️ À propos du guide
  [2] 🛠️ Outils
    [2.1] 🔊 NoiseModelling
    [2.2] 💽 Bases de données
    [2.3] 🖥️ Serveur de calcul
  [3] 🏘️ ​🔴 Bâtiments
    ✓ 2 émoji(s) dans le contenu [markdown]
    [3.1] 🙎‍♂️ ​Affectation des populations
    ✓ 1 émoji(s) dans le contenu [markdown]
    [3.2] 🧑‍🔧 Implémentation - Bâti
    ✓ 2 émoji(s) dans le contenu [markdown]
  [4] 📍 Récepteurs
    [4.1] 👩‍👦 ​🔵 Décompte de population
    ✓ 1 émoji(s) dans le contenu [markdown]
    [4.2] 🎨 Visuel - isosurfaces
    ✓ 1 émoji(s) dans le contenu [markdown]
    [4.3] 🧑‍🔧 ​🔵 Implémentation - Récepteurs
  [5] 🚙 Routier
    [5.1] 🌐 ​🔴 Géométrie
    ✓ 1 émoji(s) dans le contenu [markdown]
    [5.2] 📊 ​🔴 Débits
    [5.3] ⏩ ​Vit

## 4. Aperçu des records

In [7]:
import pandas as pd

df = pd.DataFrame([r['fields'] for r in records])

# Aperçu général avec les nouvelles colonnes émoji
cols = ['numero', 'niveau', 'emoji', 'titre_propre', 'url', 'contenu_format', 'contenu']
available = [c for c in cols if c in df.columns]
df[available].head(20)

,numero,niveau,emoji,titre_propre,url,contenu_format,contenu
0,1,1,🌟,Guide,https://docs.numerique.gouv.fr/docs/242c7d7f-b...,markdown,Ce guide méthodologique est construit autour d...
1,1.1,2,📜,​Rapportage,https://docs.numerique.gouv.fr/docs/d3521e51-f...,markdown,## Indicateurs\n\nLa liste des indicateurs att...
2,1.2,2,🛣️,GITT,https://docs.numerique.gouv.fr/docs/e85bb5f4-5...,markdown,"À chaque nouvelle échéance, l'état est en char..."
3,1.3,2,🗺️,CBS,https://docs.numerique.gouv.fr/docs/3c2d989c-4...,markdown,"Au regard de la réglementation, il existe 4 ty..."
4,1.4,2,🔠,Glossaire,https://docs.numerique.gouv.fr/docs/6b70ef38-7...,markdown,* **CBS** : Carte de Bruit Stratégique\n\n* **...
5,1.5,2,📚,Bibliographie associée,https://docs.numerique.gouv.fr/docs/e8d2b52d-d...,markdown,Common Noise Assessment Methods in Europe (CNO...
6,1.6,2,,ℹ️ À propos du guide,https://docs.numerique.gouv.fr/docs/bae48920-3...,markdown,## Auteurs :\n\n* Gwendall Petit ([UMRAE](http...
7,2,1,🛠️,Outils,https://docs.numerique.gouv.fr/docs/1755bc0c-5...,markdown,Dans cette section nous listons les outils gra...
8,2.1,2,🔊,NoiseModelling,https://docs.numerique.gouv.fr/docs/9ee6bf02-1...,markdown,![\_images/NoiseModelling\_banner.png](https:/...
9,2.2,2,💽,Bases de données,https://docs.numerique.gouv.fr/docs/17dcf47e-1...,markdown,## PostGreSQL / PostGIS\n\nL'ensemble des donn...


## 5. Envoi vers Grist
⚠️ Lance cette cellule uniquement quand les records semblent corrects.

In [8]:

from grist_client import GristClient
import requests

grist = GristClient()

# ── Colonnes existantes dans la table Grist ───────────────────────────────────
GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}
# GRIST_COLUMNS = None  # ← décommente pour envoyer tous les champs sans filtrage

# ── Taille maximale du champ contenu (caractères) ────────────────────────────
MAX_CONTENU = 100_000

def filter_record(record):
    fields = {k: v for k, v in record['fields'].items()
              if GRIST_COLUMNS is None or k in GRIST_COLUMNS}
    if 'contenu' in fields and isinstance(fields['contenu'], str):
        if len(fields['contenu']) > MAX_CONTENU:
            fields['contenu'] = fields['contenu'][:MAX_CONTENU] + '\n[…contenu tronqué]'
    return {'fields': fields}

ok, ko = [], []

for i, record in enumerate(records, start=1):
    titre = record['fields'].get('titre') or record['fields'].get('titre_propre', '?')
    filtered = filter_record(record)
    contenu_len = len(filtered['fields'].get('contenu', ''))
    try:
        grist.add_records('Chapitres', [filtered])
        ok.append(i)
        print(f"  [{i:>3}/{len(records)}] ✓  {titre}  ({contenu_len} car.)")
    except requests.HTTPError as e:
        body = e.response.text if e.response is not None else '(pas de réponse)'
        ko.append({'index': i, 'titre': titre, 'erreur': str(e), 'body': body, 'record': record})
        print(f"  [{i:>3}/{len(records)}] ✗  {titre}  ({contenu_len} car.)")
        print(f"          → {e}")
        print(f"          Corps Grist : {body[:300]}")
    except Exception as e:
        ko.append({'index': i, 'titre': titre, 'erreur': str(e), 'body': '', 'record': record})
        print(f"  [{i:>3}/{len(records)}] ✗  {titre}")
        print(f"          → {e}")

print(f"\n── Résumé ──────────────────────────────────────────")
print(f"  Succès : {len(ok)}")
print(f"  Échecs : {len(ko)}")
if ko:
    print("\n  Détail des échecs :")
    for item in ko:
        print(f"    [{item['index']}] {item['titre']!r} → {item['erreur']}")
        if item.get('body'):
            print(f"             Corps : {item['body'][:200]}")


  [  1/50] ✓  🌟 Guide  (304 car.)
  [  2/50] ✓  📜 ​Rapportage  (3614 car.)
  [  3/50] ✓  🛣️ GITT  (3803 car.)
  [  4/50] ✓  🗺️ CBS  (814 car.)
  [  5/50] ✓  🔠 Glossaire  (1207 car.)
  [  6/50] ✓  📚 Bibliographie associée  (442 car.)
  [  7/50] ✓  ℹ️ À propos du guide  (339 car.)
  [  8/50] ✓  🛠️ Outils  (245 car.)
  [  9/50] ✓  🔊 NoiseModelling  (699 car.)
  [ 10/50] ✓  💽 Bases de données  (937 car.)
  [ 11/50] ✓  🖥️ Serveur de calcul  (954 car.)
  [ 12/50] ✓  🏘️ ​🔴 Bâtiments  (6450 car.)
  [ 13/50] ✓  🙎‍♂️ ​Affectation des populations  (2989 car.)
  [ 14/50] ✓  🧑‍🔧 Implémentation - Bâti  (4406 car.)
  [ 15/50] ✓  📍 Récepteurs  (1266 car.)
  [ 16/50] ✓  👩‍👦 ​🔵 Décompte de population  (6414 car.)
  [ 17/50] ✓  🎨 Visuel - isosurfaces  (2971 car.)
  [ 18/50] ✓  🧑‍🔧 ​🔵 Implémentation - Récepteurs  (6010 car.)
  [ 19/50] ✓  🚙 Routier  (489 car.)
  [ 20/50] ✓  🌐 ​🔴 Géométrie  (1552 car.)
  [ 21/50] ✓  📊 ​🔴 Débits  (6613 car.)
  [ 22/50] ✓  ⏩ ​Vitesses  (1892 car.)
  [ 23/50] ✓  ✖️ Intersecti

In [ ]:

# ── Envoi individuel d'un record (debug — cellule autonome) ──────────────────
import requests
from grist_client import GristClient
from docs_client import DocsClient

INDEX = 21  # numéro du record à tester (base 1)

_GRIST_COLUMNS = {'titre', 'niveau', 'ordre', 'numero', 'url', 'contenu'}
_MAX_CONTENU = 100_000
_WAF_SQL_LANGUAGES = ['sql', 'postgresql', 'postgres', 'psql', 'plpgsql',
                      'mysql', 'sqlite', 'tsql', 'mariadb']

def _filter_record(record):
    fields = {k: v for k, v in record['fields'].items() if k in _GRIST_COLUMNS}
    if 'contenu' in fields and isinstance(fields['contenu'], str):
        fields['contenu'] = DocsClient.strip_code_blocks(
            fields['contenu'], languages=_WAF_SQL_LANGUAGES
        )
        if len(fields['contenu']) > _MAX_CONTENU:
            fields['contenu'] = fields['contenu'][:_MAX_CONTENU] + '\n[…contenu tronqué]'
    return {'fields': fields}

_grist = GristClient()
record_n = records[INDEX - 1]
filtered_n = _filter_record(record_n)

print(f"Champs envoyés : {list(filtered_n['fields'].keys())}")
print(f"Taille contenu : {len(filtered_n['fields'].get('contenu', ''))} car.")
print(f"Titre          : {filtered_n['fields'].get('titre')!r}")
print()

try:
    result = _grist.add_records('Chapitres', [filtered_n])
    print(f"✓ Record {INDEX} ajouté. Réponse : {result}")
except requests.HTTPError as e:
    body = e.response.text if e.response is not None else '(pas de réponse)'
    print(f"✗ HTTPError : {e}")
    print(f"  Corps Grist : {body}")
except Exception as e:
    print(f"✗ Erreur : {e}")


Champs envoyés : ['titre', 'niveau', 'ordre', 'numero', 'url', 'contenu']
Taille contenu : 6613 car.
Titre          : '📊 \u200b🔴 Débits'

✓ Record 21 ajouté. Réponse : {'records': [{'id': 50}]}
